# Single-τ DAS-FedAvg Run (GPU-Friendly)

This notebook runs one τ configuration of `federated_das.py` on a Kaggle/Colab GPU runtime.
It clones your working branch, installs dependencies, runs the validation-selected DAS-FedAvg experiment, and packages outputs for download.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Abishek-Chakravarthy/fed-rag.git"
REPO_BRANCH = "das-fedrag"

TAU = 0.0
SEED = 42
ROUNDS = 4
LOCAL_EPOCHS = 1
TARGET = "nfcorpus"

WORKSPACE = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
REPO_DIR = WORKSPACE / "fed-rag"
EXP_DIR = REPO_DIR / "zz_coderuns" / "das_fedrag"
EXPORT_DIR = WORKSPACE / f"das_fedavg_exports_{TARGET.replace('-', '_')}"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"WORKSPACE: {WORKSPACE}")
print(f"REPO_DIR : {REPO_DIR}")
print(f"EXP_DIR  : {EXP_DIR}")

In [ ]:
import os
import shutil
import subprocess
import sys

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    [
        "git", "clone", "--depth", "1",
        "--branch", REPO_BRANCH, "--single-branch",
        REPO_URL, str(REPO_DIR),
    ],
    check=True,
)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "accelerate", "datasets<3.0.0", "flwr", "pyarrow", "pydantic",
    "pydantic-settings", "transformers==4.48.0", "sentence-transformers==3.4.1",
    "peft", "matplotlib", "pandas", "tqdm",
], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "--no-deps"], check=True)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
print("Clone + install complete")

In [ ]:
import torch

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("mps available :", torch.backends.mps.is_available())

if torch.cuda.is_available():
    print("gpu device:", torch.cuda.get_device_name(0))
    import subprocess
    subprocess.run(["nvidia-smi"])
elif torch.backends.mps.is_available():
    print("Using Apple Metal backend")
else:
    print("WARNING: No GPU backend detected.")

In [ ]:
def build_run_slug(tau, seed, rounds, local_epochs, target):
    target_slug = target.replace("-", "_")
    return f"tau_{tau:.2f}_seed_{seed}_target_{target_slug}_r{rounds}_e{local_epochs}"

RUN_SLUG = build_run_slug(TAU, SEED, ROUNDS, LOCAL_EPOCHS, TARGET)
print(RUN_SLUG)

In [ ]:
import re
import subprocess
import sys
from tqdm.auto import tqdm

cmd = [
    sys.executable, "-u", "federated_das.py",
    "--tau", str(TAU),
    "--seed", str(SEED),
    "--rounds", str(ROUNDS),
    "--local-epochs", str(LOCAL_EPOCHS),
    "--target", TARGET,
]

print("Running:", " ".join(cmd))
round_pattern = re.compile(r"Round\s+(\d+)\s+\|")
progress = tqdm(total=ROUNDS, desc=f"target={TARGET} tau={TAU:.2f}", unit="round")

process = subprocess.Popen(
    cmd, cwd=EXP_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)

for line in process.stdout:
    print(line, end="")
    match = round_pattern.search(line)
    if match:
        progress.n = min(int(match.group(1)), ROUNDS)
        progress.refresh()

process.wait()
if process.returncode == 0:
    progress.n = ROUNDS
    progress.refresh()
progress.close()

if process.returncode != 0:
    raise subprocess.CalledProcessError(process.returncode, cmd)

In [ ]:
import json
import pandas as pd
import shutil
import zipfile

csv_path = EXP_DIR / "output_csv_files" / f"results_{RUN_SLUG}.csv"
manifest_path = EXP_DIR / "output_csv_files" / f"manifest_{RUN_SLUG}.json"
acceptance_path = EXP_DIR / "output_csv_files" / f"acceptance_{RUN_SLUG}.json"
log_path = EXP_DIR / "output_log_files" / f"log_{RUN_SLUG}.log"

for path in [csv_path, manifest_path, acceptance_path, log_path]:
    print(path.name, path.exists())

run_export_dir = EXPORT_DIR / RUN_SLUG
run_export_dir.mkdir(parents=True, exist_ok=True)

for path in [csv_path, manifest_path, acceptance_path, log_path]:
    if path.exists():
        shutil.copy2(path, run_export_dir / path.name)

zip_path = EXPORT_DIR / f"{RUN_SLUG}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in run_export_dir.iterdir():
        zf.write(file_path, arcname=file_path.name)

print(f"\nExport folder: {run_export_dir}")
print(f"Zip file     : {zip_path}")

df = pd.read_csv(csv_path)
display(df)

with open(acceptance_path, "r", encoding="utf-8") as f:
    acceptance = json.load(f)
print("\nAcceptance report:")
print(json.dumps(acceptance, indent=2))

## Download

Suggested first pass:
- run one notebook per τ value, keeping `SEED=42`, `ROUNDS=4`, `LOCAL_EPOCHS=1`, and `TARGET="nfcorpus"`
- collect the zipped outputs for each τ and compile them locally

Download either:
- the `.zip` file shown above, or
- the whole `das_fedavg_exports/<run_slug>/` folder

After collecting all τ runs, use `compile_tau_results.py` locally to generate summary plots.